# Modelamiento integral — Bank Marketing

## Despliegue de Soluciones Analíticas

**Objetivo:** comparar nueve configuraciones de clasificación, seleccionar un modelo orientado a priorización comercial, registrar los experimentos y artefactos en MLflow y guardar el modelo final.

### Antes de ejecutar
- Iniciar el servidor MLflow configurado en `http://127.0.0.1:5000`.

### Criterios metodológicos
- Se crea una partición interna de entrenamiento/validación a partir de `X_train` para los registros individuales de variantes.
- La comparación principal usa validación cruzada agrupada de cinco folds y predicciones out-of-fold (OOF).
- SMOTE y la ponderación se aplican exclusivamente al entrenamiento de cada fold.
- El test se evalúa al final. Este conjunto tuvo exposición histórica en evaluaciones anteriores del proyecto; por tanto, no se presenta como test completamente inédito.
- Las matrices cargadas ya están transformadas. Este notebook no reconstruye el preprocesamiento desde datos crudos; eso debe integrarse posteriormente en el artefacto de despliegue.


## 0. Importaciones y configuración general


In [19]:
from pathlib import Path
import json
import warnings

import joblib
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd

from imblearn.over_sampling import SMOTE
from sklearn.base import clone
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

## 0. CONFIGURACIÓN GENERAL Y RUTAS


In [20]:
RANDOM_STATE = 42
N_SPLITS = 5
TEST_SIZE_VALIDACION = 0.20
UMBRAL_REFERENCIA = 0.50
UMBRALES = np.round(np.arange(0.10, 1.00, 0.10), 2)
CAPACIDADES = (0.10, 0.20)

MLFLOW_TRACKING_URI = "http://127.0.0.1:5000"
EXPERIMENTO = "bank-marketing-modelamiento-integral"

# Configurar MLflow antes de registrar cualquier ejecución
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

print("MLflow Tracking URI:", mlflow.get_tracking_uri())

experimento_principal = mlflow.set_experiment(EXPERIMENTO)

print("Experimento principal:", experimento_principal.name)
print("Experiment ID:", experimento_principal.experiment_id)

def localizar_datos_procesados():
    """Busca los CSV de particiones existentes; no los vuelve a crear."""
    cwd = Path.cwd().resolve()
    candidatos = []

    for base in [cwd, *cwd.parents]:
        candidatos.extend([
            base / "data" / "processed",
            base / "notebooks" / "data" / "processed",
        ])

    requeridos = {
        "X_train_final.csv",
        "X_test_final.csv",
        "y_train.csv",
        "y_test.csv",
    }

    for carpeta in candidatos:
        if carpeta.exists() and requeridos.issubset(
            {p.name for p in carpeta.iterdir() if p.is_file()}
        ):
            return carpeta

    raise FileNotFoundError(
        "No se encontró una carpeta data/processed con los cuatro CSV "
        "de particiones. Actualiza el repositorio y verifica la ruta."
    )


RUTA_PROCESADOS = localizar_datos_procesados()
print("Archivos de particiones:", RUTA_PROCESADOS)

MLflow Tracking URI: http://127.0.0.1:5000
Experimento principal: bank-marketing-modelamiento-integral
Experiment ID: 3
Archivos de particiones: D:\Cris\02. Maestria en Inteligencia Analítica de Datos\0. Clases\06. Ciclo 6\Despliegue de soluciones analíticas\Proyecto\notebooks\data\processed


## 1. CARGAR PARTICIONES EXISTENTES DESDE EL REPOSITORIO


In [21]:
# index_col=0 recupera el índice original exportado desde el EDA.
X_train = pd.read_csv(
    RUTA_PROCESADOS / "X_train_final.csv",
    index_col=0,
)
X_test = pd.read_csv(
    RUTA_PROCESADOS / "X_test_final.csv",
    index_col=0,
)
y_train = pd.read_csv(
    RUTA_PROCESADOS / "y_train.csv",
    index_col=0,
).squeeze("columns").astype(int)
y_test = pd.read_csv(
    RUTA_PROCESADOS / "y_test.csv",
    index_col=0,
).squeeze("columns").astype(int)

print("\nDIMENSIONES DE LAS PARTICIONES EXISTENTES")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test: ", X_test.shape)
print("y_test: ", y_test.shape)

assert isinstance(y_train, pd.Series)
assert isinstance(y_test, pd.Series)
assert X_train.index.equals(y_train.index), "Índices train desalineados."
assert X_test.index.equals(y_test.index), "Índices test desalineados."
assert X_train.columns.equals(X_test.columns), "Columnas train/test distintas."
assert len(X_train) == len(y_train)
assert len(X_test) == len(y_test)
assert set(y_train.unique()).issubset({0, 1})
assert set(y_test.unique()).issubset({0, 1})

columnas_indice = [
    c for c in X_train.columns
    if str(c).lower().startswith("unnamed")
    or str(c).lower() == "index"
]
if columnas_indice:
    raise ValueError(
        f"X contiene columnas que parecen índices: {columnas_indice}. "
        "Revisa cómo se exportaron las matrices desde el EDA."
    )

if X_train.isna().any().any() or X_test.isna().any().any():
    raise ValueError("Hay valores ausentes en las matrices procesadas.")

print("\nDistribución de clases train:")
print(y_train.value_counts().sort_index())
print(y_train.value_counts(normalize=True).sort_index().round(4))
print("\nDistribución de clases test:")
print(y_test.value_counts().sort_index())
print("\nValidación de estructura superada.")


DIMENSIONES DE LAS PARTICIONES EXISTENTES
X_train: (36168, 94)
y_train: (36168,)
X_test:  (9043, 94)
y_test:  (9043,)

Distribución de clases train:
y
0    31937
1     4231
Name: count, dtype: int64
y
0    0.883
1    0.117
Name: proportion, dtype: float64

Distribución de clases test:
y
0    7985
1    1058
Name: count, dtype: int64

Validación de estructura superada.


## 2. PARTICIÓN INTERNA TRAIN / VALIDACIÓN


In [22]:
# No se recrea el train/test original. Se separa un 20% de X_train
# para seleccionar modelos y umbrales sin utilizar X_test.
indices_train_model, indices_val = train_test_split(
    X_train.index,
    test_size=TEST_SIZE_VALIDACION,
    random_state=RANDOM_STATE,
    stratify=y_train,
)

X_train_model = X_train.loc[indices_train_model].copy()
X_val = X_train.loc[indices_val].copy()
y_train_model = y_train.loc[indices_train_model].copy()
y_val = y_train.loc[indices_val].copy()

print("\nPARTICIÓN INTERNA")
print("Entrenamiento interno:", X_train_model.shape)
print("Validación:", X_val.shape)
print("Positivos train interno:", f"{y_train_model.mean():.2%}")
print("Positivos validación:", f"{y_val.mean():.2%}")

assert X_train_model.index.equals(y_train_model.index)
assert X_val.index.equals(y_val.index)
assert set(X_train_model.index).isdisjoint(set(X_val.index))
assert set(X_train_model.index).isdisjoint(set(X_test.index))
assert set(X_val.index).isdisjoint(set(X_test.index))


PARTICIÓN INTERNA
Entrenamiento interno: (28934, 94)
Validación: (7234, 94)
Positivos train interno: 11.70%
Positivos validación: 11.69%


## 3. FUNCIONES DE MÉTRICAS


In [23]:
def calcular_metricas_ranking(y_real, probabilidades, k):
    y_array = np.asarray(y_real)
    proba = np.asarray(probabilidades)
    n_top = max(1, int(np.ceil(len(y_array) * k)))
    posiciones = np.argsort(-proba)[:n_top]
    precision_k = float(y_array[posiciones].mean())
    prevalencia = float(y_array.mean())
    lift_k = precision_k / prevalencia if prevalencia > 0 else np.nan
    return precision_k, float(lift_k), n_top


def calcular_metricas_completas(y_real, probabilidades, umbral=0.50):
    y_array = np.asarray(y_real).astype(int)
    proba = np.asarray(probabilidades)
    pred = (proba >= umbral).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_array, pred, labels=[0, 1]
    ).ravel()

    p10, l10, n10 = calcular_metricas_ranking(y_array, proba, 0.10)
    p20, l20, n20 = calcular_metricas_ranking(y_array, proba, 0.20)

    return {
        "precision": float(precision_score(y_array, pred, zero_division=0)),
        "recall": float(recall_score(y_array, pred, zero_division=0)),
        "f1": float(f1_score(y_array, pred, zero_division=0)),
        "pr_auc": float(average_precision_score(y_array, proba)),
        "roc_auc": float(roc_auc_score(y_array, proba)),
        "precision_at_10": p10,
        "lift_at_10": l10,
        "precision_at_20": p20,
        "lift_at_20": l20,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "positivos_predichos": int(pred.sum()),
        "porcentaje_seleccionado": float(pred.mean()),
        "clientes_top_10": int(n10),
        "clientes_top_20": int(n20),
    }

## 4. DEFINIR LOS NUEVE MODELOS CANDIDATOS


In [24]:
def crear_modelos(ratio_positivo=None):
    """Crea instancias nuevas para cada fold o entrenamiento final."""
    if ratio_positivo is None:
        ratio_positivo = (
            (y_train == 0).sum() / max((y_train == 1).sum(), 1)
        )

    return {
        "reglog_sin_balanceo": Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(
                max_iter=2000,
                random_state=RANDOM_STATE,
            )),
        ]),

        "reglog_balanceada": Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=RANDOM_STATE,
            )),
        ]),

        "random_forest_sin_balanceo": RandomForestClassifier(
            n_estimators=300,
            max_depth=None,
            min_samples_leaf=2,
            class_weight=None,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),

        "random_forest_balanceado": RandomForestClassifier(
            n_estimators=300,
            max_depth=None,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),

        "gradient_boosting": GradientBoostingClassifier(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=3,
            random_state=RANDOM_STATE,
        ),

        "gradient_boosting_ponderado": GradientBoostingClassifier(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=3,
            random_state=RANDOM_STATE,
        ),

        "gradient_boosting_smote": GradientBoostingClassifier(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=3,
            random_state=RANDOM_STATE,
        ),

        "xgboost_sin_balanceo": XGBClassifier(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=3,
            random_state=RANDOM_STATE,
            objective="binary:logistic",
            eval_metric="logloss",
            scale_pos_weight=1,
            n_jobs=-1,
        ),

        "xgboost_ponderado": XGBClassifier(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=3,
            random_state=RANDOM_STATE,
            objective="binary:logistic",
            eval_metric="logloss",
            scale_pos_weight=ratio_positivo,
            n_jobs=-1,
        ),
    }


def ajustar_modelo(nombre, modelo, X, y):
    """Aplica balanceo exclusivamente sobre el fold de entrenamiento."""
    if nombre == "gradient_boosting_ponderado":
        pesos = compute_sample_weight(
            class_weight="balanced",
            y=np.asarray(y),
        )
        modelo.fit(X, y, sample_weight=pesos)

    elif nombre == "gradient_boosting_smote":
        smote = SMOTE(random_state=RANDOM_STATE)
        X_res, y_res = smote.fit_resample(X, y)
        modelo.fit(X_res, y_res)

    elif nombre == "xgboost_ponderado":
        conteos = pd.Series(y).value_counts()
        ratio = float(conteos.get(0, 0) / max(conteos.get(1, 1), 1))
        modelo.set_params(scale_pos_weight=ratio)
        modelo.fit(X, y)

    else:
        modelo.fit(X, y)

    return modelo


modelos_base = crear_modelos()
print("\nModelos candidatos:", list(modelos_base.keys()))


Modelos candidatos: ['reglog_sin_balanceo', 'reglog_balanceada', 'random_forest_sin_balanceo', 'random_forest_balanceado', 'gradient_boosting', 'gradient_boosting_ponderado', 'gradient_boosting_smote', 'xgboost_sin_balanceo', 'xgboost_ponderado']


## 5. VALIDACIÓN CRUZADA AGRUPADA (5 FOLDS)


In [25]:
# Agrupa filas con el mismo perfil de predictores para evitar que
# un perfil repetido aparezca simultáneamente en train y validación.
hash_train = pd.util.hash_pandas_object(X_train, index=False)
groups_train = pd.factorize(hash_train)[0]

cv_agrupada = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

resultados_folds = []
probabilidades_oof = {
    nombre: np.full(len(X_train), np.nan)
    for nombre in modelos_base
}

for fold, (idx_fit, idx_valid) in enumerate(
    cv_agrupada.split(X_train, y_train, groups_train),
    start=1,
):
    X_fold_train = X_train.iloc[idx_fit]
    X_fold_val = X_train.iloc[idx_valid]
    y_fold_train = y_train.iloc[idx_fit]
    y_fold_val = y_train.iloc[idx_valid]

    grupos_fit = set(groups_train[idx_fit])
    grupos_valid = set(groups_train[idx_valid])
    assert grupos_fit.isdisjoint(grupos_valid)

    print(f"\nFOLD {fold}/{N_SPLITS}")

    for nombre in modelos_base:
        modelo = crear_modelos()[nombre]
        modelo = ajustar_modelo(
            nombre, modelo, X_fold_train, y_fold_train
        )

        proba = modelo.predict_proba(X_fold_val)[:, 1]
        probabilidades_oof[nombre][idx_valid] = proba

        metricas = calcular_metricas_completas(
            y_fold_val, proba, UMBRAL_REFERENCIA
        )
        metricas.update({"modelo": nombre, "fold": fold})
        resultados_folds.append(metricas)

        print(
            f"{nombre}: PR-AUC={metricas['pr_auc']:.4f}; "
            f"Lift@10={metricas['lift_at_10']:.4f}"
        )

for nombre, proba in probabilidades_oof.items():
    if np.isnan(proba).any():
        raise AssertionError(f"Predicciones OOF incompletas: {nombre}")

resultados_cv_agrupada = pd.DataFrame(resultados_folds)

metricas_cv = [
    "precision", "recall", "f1", "pr_auc", "roc_auc",
    "precision_at_10", "lift_at_10",
    "precision_at_20", "lift_at_20",
]

resumen_cv = (
    resultados_cv_agrupada
    .groupby("modelo")[metricas_cv]
    .agg(["mean", "std"])
)

print("\nRESUMEN DE VALIDACIÓN CRUZADA AGRUPADA")
print(resumen_cv.round(4))


FOLD 1/5
reglog_sin_balanceo: PR-AUC=0.3542; Lift@10=3.6376
reglog_balanceada: PR-AUC=0.3524; Lift@10=3.6376
random_forest_sin_balanceo: PR-AUC=0.3360; Lift@10=3.4841
random_forest_balanceado: PR-AUC=0.3289; Lift@10=3.3188
gradient_boosting: PR-AUC=0.3683; Lift@10=3.7085
gradient_boosting_ponderado: PR-AUC=0.3640; Lift@10=3.6376
gradient_boosting_smote: PR-AUC=0.3426; Lift@10=3.6967
xgboost_sin_balanceo: PR-AUC=0.3637; Lift@10=3.7203
xgboost_ponderado: PR-AUC=0.3595; Lift@10=3.7203

FOLD 2/5
reglog_sin_balanceo: PR-AUC=0.3773; Lift@10=3.7316
reglog_balanceada: PR-AUC=0.3773; Lift@10=3.7198
random_forest_sin_balanceo: PR-AUC=0.3754; Lift@10=3.6844
random_forest_balanceado: PR-AUC=0.3587; Lift@10=3.5191
gradient_boosting: PR-AUC=0.3883; Lift@10=3.9324
gradient_boosting_ponderado: PR-AUC=0.3888; Lift@10=3.8261
gradient_boosting_smote: PR-AUC=0.3795; Lift@10=3.6962
xgboost_sin_balanceo: PR-AUC=0.3910; Lift@10=3.9442
xgboost_ponderado: PR-AUC=0.3888; Lift@10=3.8379

FOLD 3/5
reglog_sin_bal

## 6. EVALUACIÓN DE UMBRALES CON PREDICCIONES OOF


In [26]:
resultados_umbrales = []
y_train_array = y_train.to_numpy()

for nombre, proba in probabilidades_oof.items():
    for umbral in UMBRALES:
        pred = (proba >= umbral).astype(int)
        resultados_umbrales.append({
            "modelo": nombre,
            "umbral": float(umbral),
            "precision": float(
                precision_score(y_train_array, pred, zero_division=0)
            ),
            "recall": float(
                recall_score(y_train_array, pred, zero_division=0)
            ),
            "f1": float(
                f1_score(y_train_array, pred, zero_division=0)
            ),
            "positivos_predichos": int(pred.sum()),
            "porcentaje_seleccionado": float(pred.mean()),
        })

tabla_umbrales = pd.DataFrame(resultados_umbrales)

mejores_umbrales_f1 = (
    tabla_umbrales
    .sort_values(["modelo", "f1"], ascending=[True, False])
    .groupby("modelo")
    .head(1)
    .sort_values("f1", ascending=False)
    .reset_index(drop=True)
)

print("\nMEJOR UMBRAL POR MODELO SEGÚN F1 (OOF)")
print(mejores_umbrales_f1.round(4).to_string(index=False))


MEJOR UMBRAL POR MODELO SEGÚN F1 (OOF)
                     modelo  umbral  precision  recall     f1  positivos_predichos  porcentaje_seleccionado
       xgboost_sin_balanceo     0.2     0.4297  0.3909 0.4094                 3849                   0.1064
          gradient_boosting     0.2     0.4358  0.3812 0.4067                 3701                   0.1023
gradient_boosting_ponderado     0.6     0.3819  0.4285 0.4039                 4747                   0.1312
          xgboost_ponderado     0.6     0.3847  0.4245 0.4036                 4668                   0.1291
    gradient_boosting_smote     0.3     0.4207  0.3718 0.3947                 3739                   0.1034
        reglog_sin_balanceo     0.2     0.4203  0.3720 0.3947                 3745                   0.1035
          reglog_balanceada     0.6     0.3595  0.4361 0.3941                 5132                   0.1419
 random_forest_sin_balanceo     0.3     0.4415  0.3271 0.3758                 3135              

## 7A. Registro individual de las cuatro variantes de Juan en MLflow

Esta sección complementa la comparación OOF: entrena cada variante sobre `X_train_model`, la evalúa sobre `X_val` y registra **métricas, parámetros y artefacto del modelo** en el experimento `bank-marketing-boosting-balanceo`. El conjunto de validación no se utiliza para ajustar los modelos.


In [27]:
# Variantes incorporadas por Juan: Gradient Boosting ponderado,
# Gradient Boosting con SMOTE, XGBoost base y XGBoost ponderado.
variantes_juan = [
    "gradient_boosting_ponderado",
    "gradient_boosting_smote",
    "xgboost_sin_balanceo",
    "xgboost_ponderado",
]

experimento_anterior = mlflow.get_experiment_by_name(
    "bank-marketing-boosting-balanceo"
)
if experimento_anterior is None:
    experimento_juan = mlflow.set_experiment(
        "bank-marketing-boosting-balanceo"
    )
else:
    experimento_juan = mlflow.set_experiment(
        "bank-marketing-boosting-balanceo"
    )

metricas_validacion_juan = {}
modelos_juan_entrenados = {}

for nombre in variantes_juan:
    modelo = crear_modelos()[nombre]
    modelo = ajustar_modelo(
        nombre,
        modelo,
        X_train_model,
        y_train_model,
    )

    probabilidades_val = modelo.predict_proba(X_val)[:, 1]
    metricas = calcular_metricas_completas(
        y_val,
        probabilidades_val,
        umbral=UMBRAL_REFERENCIA,
    )

    # No registrar conteos como métricas MLflow: se conservan en la tabla.
    metricas_mlflow = {
        clave: float(valor)
        for clave, valor in metricas.items()
        if isinstance(valor, (int, float, np.integer, np.floating))
        and np.isfinite(valor)
    }

    with mlflow.start_run(run_name=f"juan_{nombre}"):
        mlflow.log_params({
            "modelo": nombre,
            "evaluacion": "validacion_holdout_interna",
            "threshold": UMBRAL_REFERENCIA,
            "random_state": RANDOM_STATE,
            "n_train_interno": len(X_train_model),
            "n_validacion": len(X_val),
            "tecnica_balanceo": (
                "sample_weight" if nombre == "gradient_boosting_ponderado"
                else "SMOTE" if nombre == "gradient_boosting_smote"
                else "scale_pos_weight" if nombre == "xgboost_ponderado"
                else "ninguna"
            ),
        })
        mlflow.log_metrics(metricas_mlflow)

        if isinstance(modelo, XGBClassifier):
            mlflow.xgboost.log_model(
                xgb_model=modelo,
                name="modelo",
            )
        else:
            mlflow.sklearn.log_model(
                sk_model=modelo,
                name="modelo",
                skops_trusted_types=["sklearn.tree._tree.Tree"],
            )

    metricas_validacion_juan[nombre] = metricas
    modelos_juan_entrenados[nombre] = modelo

    print(
        f"Registrado en MLflow: {nombre} | "
        f"PR-AUC={metricas['pr_auc']:.4f} | "
        f"Lift@10={metricas['lift_at_10']:.4f}"
    )

tabla_validacion_juan = (
    pd.DataFrame.from_dict(metricas_validacion_juan, orient="index")
    .sort_values("lift_at_10", ascending=False)
)
print("\nMÉTRICAS DE VALIDACIÓN DE LAS CUATRO VARIANTES DE JUAN")
display(tabla_validacion_juan.round(4))

# Volver al experimento principal para el resto del flujo.
mlflow.set_experiment(EXPERIMENTO)

🏃 View run juan_gradient_boosting_ponderado at: http://127.0.0.1:5000/#/experiments/2/runs/1a587f3087ff4287a278e137520a0d32
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
Registrado en MLflow: gradient_boosting_ponderado | PR-AUC=0.3934 | Lift@10=3.9329
🏃 View run juan_gradient_boosting_smote at: http://127.0.0.1:5000/#/experiments/2/runs/f2319291788e441fa0e62039c1955b01
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
Registrado en MLflow: gradient_boosting_smote | PR-AUC=0.3812 | Lift@10=3.8030
🏃 View run juan_xgboost_sin_balanceo at: http://127.0.0.1:5000/#/experiments/2/runs/1ed148b2df0a475b8d25e00fc3a51bc4
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
Registrado en MLflow: xgboost_sin_balanceo | PR-AUC=0.3967 | Lift@10=3.9920
🏃 View run juan_xgboost_ponderado at: http://127.0.0.1:5000/#/experiments/2/runs/9f3120950bf04fd2b85e04052b75af72
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
Registrado en MLflow: xgboost_ponderado | PR-AU

,precision,recall,f1,pr_auc,roc_auc,precision_at_10,lift_at_10,precision_at_20,lift_at_20,tn,fp,fn,tp,positivos_predichos,porcentaje_seleccionado,clientes_top_10,clientes_top_20
xgboost_sin_balanceo,0.6957,0.1702,0.2735,0.3967,0.7480,0.4669,3.9920,0.3165,2.7065,6325,63,702,144,207,0.0286,724,1447
xgboost_ponderado,0.2734,0.5839,0.3724,0.3939,0.7493,0.4641,3.9683,0.3117,2.6651,5075,1313,352,494,1807,0.2498,724,1447
gradient_boosting_ponderado,0.2744,0.5686,0.3701,0.3934,0.7482,0.4599,3.9329,0.3110,2.6592,5116,1272,365,481,1753,0.2423,724,1447
gradient_boosting_smote,0.5821,0.2305,0.3302,0.3812,0.7405,0.4448,3.8030,0.3082,2.6356,6248,140,651,195,335,0.0463,724,1447


<Experiment: artifact_location='mlflow-artifacts:/3', creation_time=1794995800543, effective_trace_archival_retention=None, experiment_id='3', last_update_time=1794995800543, lifecycle_stage='active', name='bank-marketing-modelamiento-integral', tags={}, trace_location=None, workspace='default'>

## 7. EVALUACIÓN POR CAPACIDAD COMERCIAL TOP 10% / TOP 20%


In [28]:
resultados_capacidad = []

for nombre, proba in probabilidades_oof.items():
    for capacidad in CAPACIDADES:
        precision_k, lift_k, cantidad = calcular_metricas_ranking(
            y_train, proba, capacidad
        )
        n_top = cantidad
        posiciones = np.argsort(-proba)[:n_top]
        tp = int(y_train_array[posiciones].sum())

        resultados_capacidad.append({
            "modelo": nombre,
            "capacidad": capacidad,
            "clientes_contactados": n_top,
            "verdaderos_positivos": tp,
            "falsos_positivos": int(n_top - tp),
            "precision": precision_k,
            "recall_capturado": float(tp / y_train_array.sum()),
            "lift": lift_k,
        })

tabla_capacidad = pd.DataFrame(resultados_capacidad)

print("\nRESULTADOS OOF POR CAPACIDAD")
print(tabla_capacidad.round(4).to_string(index=False))


# Métricas Top-K calculadas por fold: más robustas ante diferencias
# de escala de probabilidades entre los modelos de cada fold.
resultados_capacidad_fold = []

for fold, (idx_fit, idx_valid) in enumerate(
    cv_agrupada.split(X_train, y_train, groups_train),
    start=1,
):
    y_fold = y_train.iloc[idx_valid].to_numpy()

    for nombre, proba_oof in probabilidades_oof.items():
        proba_fold = proba_oof[idx_valid]

        for capacidad in CAPACIDADES:
            n_top = max(1, int(np.ceil(len(y_fold) * capacidad)))
            posiciones = np.argsort(-proba_fold)[:n_top]
            y_top = y_fold[posiciones]
            tp = int(y_top.sum())
            precision_k = float(y_top.mean())
            lift_k = precision_k / y_fold.mean()

            resultados_capacidad_fold.append({
                "modelo": nombre,
                "fold": fold,
                "capacidad": capacidad,
                "precision": precision_k,
                "recall_capturado": float(tp / y_fold.sum()),
                "lift": float(lift_k),
                "clientes_contactados": n_top,
                "verdaderos_positivos": tp,
            })

tabla_capacidad_fold = pd.DataFrame(resultados_capacidad_fold)
resumen_capacidad_fold = (
    tabla_capacidad_fold
    .groupby(["modelo", "capacidad"])
    .agg({
        "precision": ["mean", "std"],
        "recall_capturado": ["mean", "std"],
        "lift": ["mean", "std"],
        "clientes_contactados": "mean",
        "verdaderos_positivos": "mean",
    })
)

print("\nESTABILIDAD TOP-K POR FOLD")
print(resumen_capacidad_fold.round(4))


RESULTADOS OOF POR CAPACIDAD
                     modelo  capacidad  clientes_contactados  verdaderos_positivos  falsos_positivos  precision  recall_capturado   lift
        reglog_sin_balanceo        0.1                  3617                  1547              2070     0.4277            0.3656 3.6561
        reglog_sin_balanceo        0.2                  7234                  2154              5080     0.2978            0.5091 2.5454
          reglog_balanceada        0.1                  3617                  1536              2081     0.4247            0.3630 3.6301
          reglog_balanceada        0.2                  7234                  2142              5092     0.2961            0.5063 2.5312
 random_forest_sin_balanceo        0.1                  3617                  1494              2123     0.4130            0.3531 3.5309
 random_forest_sin_balanceo        0.2                  7234                  2056              5178     0.2842            0.4859 2.4296
   random_f

## 8. COMPARACIÓN Y SELECCIÓN DEL CANDIDATO


In [29]:
# La selección se basa en el promedio de Lift@10 por fold, no en un
# ranking global de probabilidades OOF que puede mezclar escalas entre folds.
lift10_por_modelo = (
    resumen_capacidad_fold
    .xs(0.10, level="capacidad")["lift"]
    [["mean", "std"]]
    .rename(columns={"mean": "lift10_mean", "std": "lift10_std"})
)
# Resumen de métricas promedio de los nueve modelos
metricas_cv = [
    "precision",
    "recall",
    "f1",
    "pr_auc",
    "roc_auc",
    "precision_at_10",
    "lift_at_10",
    "precision_at_20",
    "lift_at_20",
]

resumen_medias = (
    resultados_cv_agrupada
    .groupby("modelo")[metricas_cv]
    .mean()
)

print("Resumen de métricas de validación cruzada:")
display(resumen_medias.round(4))
ranking_seleccion = (
    lift10_por_modelo
    .join(resumen_medias[["pr_auc", "roc_auc"]])
    .sort_values(
        ["lift10_mean", "pr_auc"],
        ascending=False
    )
)

print("\nRANKING DE SELECCIÓN: LIFT@10 MEDIO POR FOLD Y PR-AUC")
display(ranking_seleccion.round(4))

# Criterio explícito: mayor Lift@10 promedio entre folds;
# PR-AUC promedio se utiliza como desempate.
modelo_seleccionado = ranking_seleccion.index[0]

# Umbral candidato del modelo ganador: maximiza F1 en predicciones OOF.
fila_umbral = (
    tabla_umbrales[tabla_umbrales["modelo"] == modelo_seleccionado]
    .sort_values("f1", ascending=False)
    .iloc[0]
)
umbral_final = float(fila_umbral["umbral"])

print("\nMODELO CANDIDATO:", modelo_seleccionado)
print("Umbral candidato por F1 OOF:", umbral_final)
print(
    "Nota: Top 10% y Top 20% son reglas de capacidad comercial; "
    "no son equivalentes al umbral binario."
)

Resumen de métricas de validación cruzada:


,precision,recall,f1,pr_auc,roc_auc,precision_at_10,lift_at_10,precision_at_20,lift_at_20
modelo,,,,,,,,,
gradient_boosting,0.6698,0.1742,0.2763,0.3805,0.7359,0.4401,3.7618,0.3015,2.5769
gradient_boosting_ponderado,0.2664,0.5620,0.3614,0.3772,0.7360,0.4365,3.7311,0.3019,2.5805
gradient_boosting_smote,0.5716,0.2198,0.3167,0.3650,0.7282,0.4285,3.6626,0.2962,2.5320
random_forest_balanceado,0.3277,0.4242,0.3697,0.3480,0.7069,0.3975,3.3981,0.2838,2.4257
random_forest_sin_balanceo,0.6325,0.1879,0.2895,0.3590,0.7070,0.4122,3.5233,0.2839,2.4269
reglog_balanceada,0.2486,0.5859,0.3491,0.3650,0.7342,0.4249,3.6319,0.2959,2.5297
reglog_sin_balanceo,0.6541,0.1600,0.2570,0.3660,0.7335,0.4268,3.6484,0.2969,2.5379
xgboost_ponderado,0.2699,0.5651,0.3653,0.3796,0.7368,0.4403,3.7641,0.3019,2.5805
xgboost_sin_balanceo,0.6749,0.1688,0.2698,0.3822,0.7363,0.4420,3.7783,0.3020,2.5816



RANKING DE SELECCIÓN: LIFT@10 MEDIO POR FOLD Y PR-AUC


,lift10_mean,lift10_std,pr_auc,roc_auc
modelo,,,,
xgboost_sin_balanceo,3.7783,0.1465,0.3822,0.7363
xgboost_ponderado,3.7641,0.1083,0.3796,0.7368
gradient_boosting,3.7618,0.1581,0.3805,0.7359
gradient_boosting_ponderado,3.7311,0.1490,0.3772,0.7360
gradient_boosting_smote,3.6626,0.1772,0.3650,0.7282
reglog_sin_balanceo,3.6484,0.1349,0.3660,0.7335
reglog_balanceada,3.6319,0.1357,0.3650,0.7342
random_forest_sin_balanceo,3.5233,0.1674,0.3590,0.7070
random_forest_balanceado,3.3981,0.1497,0.3480,0.7069



MODELO CANDIDATO: xgboost_sin_balanceo
Umbral candidato por F1 OOF: 0.2
Nota: Top 10% y Top 20% son reglas de capacidad comercial; no son equivalentes al umbral binario.


## 9. ENTRENAR EL MODELO SELECCIONADO CON TODO X_TRAIN


In [30]:
modelo_final = crear_modelos()[modelo_seleccionado]
modelo_final = ajustar_modelo(
    modelo_seleccionado,
    modelo_final,
    X_train,
    y_train,
)

print("\nModelo final entrenado con todo X_train.")
print("Modelo:", modelo_seleccionado)
print("Registros:", len(X_train))
print("Variables:", X_train.shape[1])


Modelo final entrenado con todo X_train.
Modelo: xgboost_sin_balanceo
Registros: 36168
Variables: 94


## 10. EVALUACIÓN FINAL EN TEST (UNA SOLA VEZ)


In [31]:
probabilidades_test = modelo_final.predict_proba(X_test)[:, 1]
metricas_test = calcular_metricas_completas(
    y_test,
    probabilidades_test,
    umbral=umbral_final,
)

print("\nEVALUACIÓN FINAL EN TEST")
print(pd.DataFrame([metricas_test]).round(4).to_string(index=False))

resultados_top_k_test = []
for capacidad in CAPACIDADES:
    precision_k, lift_k, cantidad = calcular_metricas_ranking(
        y_test, probabilidades_test, capacidad
    )
    posiciones = np.argsort(-probabilidades_test)[:cantidad]
    tp = int(y_test.to_numpy()[posiciones].sum())

    resultados_top_k_test.append({
        "capacidad": f"Top {int(capacidad * 100)}%",
        "clientes_contactados": cantidad,
        "verdaderos_positivos": tp,
        "falsos_positivos": cantidad - tp,
        "precision": precision_k,
        "recall_capturado": tp / y_test.sum(),
        "lift": lift_k,
    })

tabla_top_k_test = pd.DataFrame(resultados_top_k_test)
print("\nRESULTADOS TOP-K EN TEST")
print(tabla_top_k_test.round(4).to_string(index=False))

# Advertencia metodológica: el conjunto test ya tuvo exposición histórica
# en evaluaciones anteriores del proyecto. No describirlo como test virgen.


EVALUACIÓN FINAL EN TEST
 precision  recall     f1  pr_auc  roc_auc  precision_at_10  lift_at_10  precision_at_20  lift_at_20   tn  fp  fn  tp  positivos_predichos  porcentaje_seleccionado  clientes_top_10  clientes_top_20
     0.429  0.3913 0.4093  0.3896   0.7424           0.4398      3.7589           0.3085      2.6365 7434 551 644 414                  965                   0.1067              905             1809

RESULTADOS TOP-K EN TEST
capacidad  clientes_contactados  verdaderos_positivos  falsos_positivos  precision  recall_capturado   lift
  Top 10%                   905                   398               507     0.4398            0.3762 3.7589
  Top 20%                  1809                   558              1251     0.3085            0.5274 2.6365


## 11. REGISTRO EN MLFLOW


In [32]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENTO)

# Registrar resumen CV de cada candidato sin duplicar artefactos.
for nombre in resumen_medias.index:
    fila = resumen_medias.loc[nombre]

    with mlflow.start_run(run_name=f"cv_agrupada_{nombre}"):
        mlflow.log_params({
            "modelo": nombre,
            "validacion": "StratifiedGroupKFold",
            "n_splits": N_SPLITS,
            "random_state": RANDOM_STATE,
            "evaluacion": "OOF agrupada",
        })

        for metrica in metricas_cv:
            mlflow.log_metric(
                f"{metrica}_mean",
                float(fila[metrica]),
            )
            desviacion = (
                resultados_cv_agrupada
                .loc[resultados_cv_agrupada["modelo"] == nombre, metrica]
                .std()
            )
            mlflow.log_metric(
                f"{metrica}_std",
                float(desviacion) if pd.notna(desviacion) else 0.0,
            )

# Run del modelo final: parámetros, CV, umbral, test y artefacto.
with mlflow.start_run(run_name=f"modelo_final_{modelo_seleccionado}") as run:
    mlflow.log_params({
        "modelo_seleccionado": modelo_seleccionado,
        "umbral_final": umbral_final,
        "criterio_seleccion": "Lift@10 medio en validacion agrupada; desempate PR-AUC",
        "validacion": "StratifiedGroupKFold",
        "n_splits": N_SPLITS,
        "estrategia_comercial_principal": "Top 10%",
        "estrategia_comercial_alternativa": "Top 20%",
        "test_con_exposicion_historica": True,
    })

    for metrica in metricas_cv:
        mlflow.log_metric(
            f"cv_{metrica}",
            float(resumen_medias.loc[modelo_seleccionado, metrica]),
        )

    for metrica, valor in metricas_test.items():
        if isinstance(valor, (int, float, np.integer, np.floating)):
            mlflow.log_metric(f"test_{metrica}", float(valor))

    mlflow.set_tags({
        "tipo_modelo": "modelo_final_candidato",
        "validacion": "agrupada",
        "test": "exposicion_historica_documentada",
        "objetivo": "priorizacion_clientes_bank_marketing",
    })

    if isinstance(modelo_final, XGBClassifier):
        import mlflow.xgboost
        mlflow.xgboost.log_model(
            xgb_model=modelo_final,
            name="modelo_final",
        )
    else:
        mlflow.sklearn.log_model(
            sk_model=modelo_final,
            name="modelo_final",
            skops_trusted_types=["sklearn.tree._tree.Tree"],
        )

    run_id_final = run.info.run_id

print("\nMLflow run final:", run_id_final)

🏃 View run cv_agrupada_gradient_boosting at: http://127.0.0.1:5000/#/experiments/3/runs/d056e9f6fcb646b180c1e5b91d2751ce
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
🏃 View run cv_agrupada_gradient_boosting_ponderado at: http://127.0.0.1:5000/#/experiments/3/runs/2cc7c3c703524dad824211a6883ef093
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
🏃 View run cv_agrupada_gradient_boosting_smote at: http://127.0.0.1:5000/#/experiments/3/runs/ce96b55361b94208be35a76f0fb1add8
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
🏃 View run cv_agrupada_random_forest_balanceado at: http://127.0.0.1:5000/#/experiments/3/runs/10ad8213956049c39d32567ce2101e49
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
🏃 View run cv_agrupada_random_forest_sin_balanceo at: http://127.0.0.1:5000/#/experiments/3/runs/df0810539d0e495e891090eb59ef2b5e
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
🏃 View run cv_agrupada_reglog_balanceada at: http://127.0.0.1:5

## 12. INTERPRETABILIDAD (SI EL MODELO TIENE FEATURE_IMPORTANCES_)


In [33]:
if hasattr(modelo_final, "feature_importances_"):
    importancias = pd.Series(
        modelo_final.feature_importances_,
        index=X_train.columns,
        name="importancia",
    ).sort_values(ascending=False)

    print("\nTOP 15 VARIABLES IMPORTANTES")
    print(importancias.head(15).to_string())
else:
    importancias = None
    print(
        "\nEl modelo seleccionado no expone feature_importances_. "
        "No se calcula esta interpretación con este método."
    )


TOP 15 VARIABLES IMPORTANTES
num__previous_success               0.536114
num__active_loan_products_count     0.078423
num__is_single                      0.038285
num__age                            0.031288
num__is_married                     0.025321
num__days_since_previous_contact    0.021914
num__previous                       0.021510
cat__job_blue-collar                0.017590
num__balance                        0.016593
cat__education_tertiary             0.013378
num__has_housing_loan               0.011668
num__previous_contact_recent        0.010960
num__previous_failure               0.009954
cat__previous_contact_band_one      0.009911
num__is_student                     0.009558


## 13. GUARDAR MODELO, MÉTRICAS Y METADATOS


In [42]:
# Ubicar raíz del proyecto buscando .git.
def localizar_raiz_proyecto():
    for base in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (base / ".git").exists():
            return base
    return Path.cwd().resolve()


RAIZ_PROYECTO = localizar_raiz_proyecto()
CARPETA_MODELOS = RAIZ_PROYECTO / "models"
CARPETA_REPORTES = RAIZ_PROYECTO / "reports" / "modeling"

CARPETA_MODELOS.mkdir(parents=True, exist_ok=True)
CARPETA_REPORTES.mkdir(parents=True, exist_ok=True)

ruta_modelo = CARPETA_MODELOS / "modelo_final_seleccionado.joblib"
joblib.dump(modelo_final, ruta_modelo)

metadata = {
    "modelo_seleccionado": modelo_seleccionado,
    "umbral_final": umbral_final,
    "criterio_seleccion": "Lift@10 medio en validacion agrupada; desempate PR-AUC",
    "registros_train": int(len(X_train)),
    "registros_test": int(len(X_test)),
    "numero_variables": int(X_train.shape[1]),
    "validacion": "StratifiedGroupKFold",
    "n_splits": N_SPLITS,
    "mlflow_experiment": EXPERIMENTO,
    "mlflow_run_id": run_id_final,
    "nota_test": (
        "El conjunto test tuvo exposicion historica en evaluaciones previas; "
        "no se considera completamente inedito."
    ),
    "nota_despliegue": (
        "El modelo se entrenó con matrices ya transformadas. "
        "Para inferencia desde datos crudos se debe integrar el preprocesamiento "
        "del EDA y respetar exactamente las 94/95 columnas y su orden."
    ),
}

with open(
    CARPETA_MODELOS / "metadata_modelo_final.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(metadata, f, ensure_ascii=False, indent=4)

# Guardar tablas para el informe y la trazabilidad.
resultados_cv_agrupada.to_csv(
    CARPETA_REPORTES / "resultados_cv_agrupada.csv",
    index=False,
)
tabla_umbrales.to_csv(
    CARPETA_REPORTES / "evaluacion_umbrales_oof.csv",
    index=False,
)
tabla_capacidad.to_csv(
    CARPETA_REPORTES / "evaluacion_capacidad_oof.csv",
    index=False,
)
tabla_top_k_test.to_csv(
    CARPETA_REPORTES / "evaluacion_top_k_test.csv",
    index=False,
)
pd.DataFrame([metricas_test]).to_csv(
    CARPETA_REPORTES / "metricas_test_modelo_final.csv",
    index=False,
)
if importancias is not None:
    importancias.to_csv(
        CARPETA_REPORTES / "importancia_variables.csv",
        header=True,
    )

print("\nPROCESO FINALIZADO")
print("Modelo:", ruta_modelo)
print("Metadatos:", CARPETA_MODELOS / "metadata_modelo_final.json")
print("Reportes:", CARPETA_REPORTES)


PROCESO FINALIZADO
Modelo: D:\Cris\02. Maestria en Inteligencia Analítica de Datos\0. Clases\06. Ciclo 6\Despliegue de soluciones analíticas\Proyecto\models\modelo_final_seleccionado.joblib
Metadatos: D:\Cris\02. Maestria en Inteligencia Analítica de Datos\0. Clases\06. Ciclo 6\Despliegue de soluciones analíticas\Proyecto\models\metadata_modelo_final.json
Reportes: D:\Cris\02. Maestria en Inteligencia Analítica de Datos\0. Clases\06. Ciclo 6\Despliegue de soluciones analíticas\Proyecto\reports\modeling


In [43]:
# Inspección del modelo final registrado en MLflow

run_id = "6844cd5b26064dbc899545d7ebd55438"

run = mlflow.get_run(run_id)

print("NOMBRE:", run.data.tags.get("mlflow.runName"))
print("ESTADO:", run.info.status)

print("\n--- MÉTRICAS ---")
display(pd.DataFrame(
    run.data.metrics.items(),
    columns=["Métrica", "Valor"]
))

print("\n--- PARÁMETROS ---")
display(pd.DataFrame(
    run.data.params.items(),
    columns=["Parámetro", "Valor"]
))

print("\n--- ARTEFACTOS ---")
artefactos = mlflow.artifacts.list_artifacts(run_id)

for artefacto in artefactos:
    print(artefacto.path, "| Directorio:", artefacto.is_dir)

NOMBRE: modelo_final_xgboost_sin_balanceo
ESTADO: FINISHED

--- MÉTRICAS ---


,Métrica,Valor
0,cv_precision,0.674871
1,cv_recall,0.168753
2,cv_f1,0.269818
3,cv_pr_auc,0.382159
4,cv_roc_auc,0.736345
5,cv_precision_at_10,0.441989
6,cv_lift_at_10,3.778289
7,cv_precision_at_20,0.302004
8,cv_lift_at_20,2.581647
9,test_precision,0.429016



--- PARÁMETROS ---


,Parámetro,Valor
0,modelo_seleccionado,xgboost_sin_balanceo
1,umbral_final,0.2
2,criterio_seleccion,Lift@10 medio en validacion agrupada; desempat...
3,validacion,StratifiedGroupKFold
4,n_splits,5
5,estrategia_comercial_principal,Top 10%
6,estrategia_comercial_alternativa,Top 20%
7,test_con_exposicion_historica,True



--- ARTEFACTOS ---


In [44]:
for artefacto in mlflow.artifacts.list_artifacts(
    "6844cd5b26064dbc899545d7ebd55438"
):
    print(artefacto.path, artefacto.is_dir)

El proceso de modelamiento permitió comparar nueve configuraciones de clasificación basadas en regresión logística, Random Forest, Gradient Boosting y XGBoost, incluyendo variantes con y sin estrategias de balanceo de clases. Debido al desbalance presente en el conjunto de datos, la selección no se fundamentó exclusivamente en accuracy ni en ROC-AUC. Se utilizaron métricas complementarias como PR-AUC, precisión, recall, F1-score y, especialmente, Precision@K y Lift@K, que permiten evaluar la capacidad de los modelos para priorizar clientes con mayor probabilidad de suscripción al depósito a plazo.

La comparación se realizó mediante validación cruzada agrupada (StratifiedGroupKFold) de cinco particiones. Esta estrategia permitió controlar la distribución de registros con perfiles idénticos de variables predictoras entre entrenamiento y validación, evitando que un mismo grupo de perfiles apareciera en ambos conjuntos dentro de una misma partición. Los resultados muestran que XGBoost sin balanceo obtuvo el mejor desempeño en la priorización comercial, con un Lift@10 promedio de 3,7783 y una precisión dentro del 10 % de clientes con mayor puntuación de 0,4420. También alcanzó el mayor PR-AUC promedio, de 0,3822, y un ROC-AUC de 0,7363. Su desempeño fue ligeramente superior al de Gradient Boosting, cuyo Lift@10 fue 3,7618 y PR-AUC 0,3805. Aunque las diferencias entre ambos modelos son reducidas, XGBoost sin balanceo presentó la mejor combinación de capacidad de ordenamiento y concentración de clientes potenciales.

Las variantes ponderadas de Gradient Boosting y XGBoost mostraron un comportamiento diferente: aumentaron considerablemente el recall, con valores promedio de 0,5620 y 0,5651, respectivamente, pero redujeron la precisión. Esto significa que identifican una mayor proporción de clientes positivos, aunque también clasifican como potenciales a más clientes que finalmente no pertenecen a esa categoría. Por su parte, Random Forest balanceado obtuvo el mayor F1 promedio entre los modelos evaluados, con 0,3697, pero presentó un desempeño inferior en las métricas de priorización comercial. Por tanto, el modelo más conveniente depende del objetivo operativo: si se busca recuperar una mayor proporción de positivos, las variantes ponderadas pueden resultar interesantes; si se busca concentrar los esfuerzos comerciales en los clientes con mayor probabilidad de respuesta, XGBoost sin balanceo ofrece mejores resultados.

La evaluación de diferentes umbrales permitió comprobar que el valor convencional de 0,5 no representa necesariamente la mejor decisión. Para XGBoost sin balanceo, el umbral de 0,2 obtuvo el mayor F1-score en las predicciones fuera de muestra (out-of-fold), con 0,4094, una precisión de 0,4297 y un recall de 0,3909. Con este umbral se habría seleccionado aproximadamente el 10,64 % de los registros evaluados. En contraste, utilizar un umbral de 0,5 elevaría la precisión a 0,6723, pero reduciría el recall a 0,1688 y seleccionaría solamente el 2,94 % de los registros. Esto evidencia el intercambio entre precisión y cobertura: un umbral más alto produce una selección más restrictiva, mientras que uno más bajo permite identificar una mayor cantidad de clientes potenciales, aunque con más falsos positivos. En consecuencia, se adoptó el umbral de 0,2 para la clasificación binaria, al presentar el mayor F1-score OOF entre los umbrales evaluados. Esta decisión responde al equilibrio entre precisión y recall, pero no implica que sea necesariamente el umbral óptimo para cualquier escenario comercial.

Adicionalmente, se evaluaron dos escenarios de capacidad comercial mediante estrategias de ranking Top 10 % y Top 20 %. Estas estrategias son diferentes de la clasificación basada en un umbral fijo: en lugar de seleccionar a todos los clientes cuya probabilidad supere un valor determinado, ordenan los registros según su puntuación y seleccionan una proporción definida de los clientes. En la validación agrupada, XGBoost sin balanceo alcanzó una precisión de 0,4420 y un Lift@10 de 3,7783 al seleccionar el 10 % de los clientes. Al ampliar la capacidad al 20 %, la precisión fue de 0,3020 y el Lift@20 de 2,5816. Esto significa que, bajo el escenario Top 10 %, la proporción de clientes positivos entre los seleccionados fue aproximadamente 3,78 veces la proporción de positivos de la población evaluada. El escenario Top 20 % permite contactar más clientes y recuperar una mayor cantidad de positivos, aunque disminuye la concentración de resultados favorables. Por esta razón, el ranking Top K se considera una herramienta complementaria al umbral de clasificación: el primero permite ajustar la selección a la capacidad comercial disponible, mientras que el segundo determina la clasificación binaria de cada registro.

La evaluación final sobre el conjunto de prueba presentó una precisión de 0,4290, un recall de 0,3913 y un F1-score de 0,4093. El modelo obtuvo un PR-AUC de 0,3896 y un ROC-AUC de 0,7424. Al aplicar el umbral seleccionado de 0,2, clasificó correctamente 414 clientes positivos (verdaderos positivos) y 7.434 negativos (verdaderos negativos); adicionalmente, registró 551 falsos positivos y 644 falsos negativos. En términos operativos, seleccionó 965 registros, equivalentes al 10,67 % del conjunto de prueba. Estos resultados muestran una capacidad de discriminación y un equilibrio entre precisión y recall coherentes con el objetivo de priorización. No obstante, el conjunto de prueba tuvo exposición histórica en evaluaciones anteriores del proyecto y, por tanto, no puede considerarse completamente inédito. Esta condición debe tenerse en cuenta al interpretar las métricas y limita la posibilidad de tratarlas como una estimación independiente y definitiva del desempeño futuro.

La evaluación de capacidad comercial en test refuerza la utilidad del modelo para priorizar contactos. Al seleccionar el Top 10 %, se identificaron 905 clientes, de los cuales 398 fueron positivos, con una precisión de 0,4398 y un Lift de 3,7589. Al ampliar la selección al Top 20 %, se contactaron 1.809 clientes y se identificaron 558 positivos, con una precisión de 0,3085 y un Lift de 2,6365. Por consiguiente, el Top 10 % representa una alternativa adecuada cuando la capacidad de contacto es limitada y se busca concentrar los esfuerzos en los clientes con mayor probabilidad de respuesta. El Top 20 % constituye una alternativa cuando existe mayor disponibilidad de recursos comerciales y se desea ampliar la cobertura, aceptando una menor precisión. Estos escenarios no deben confundirse con la cantidad de registros seleccionados mediante el umbral de 0,2, ya que responden a reglas de selección diferentes.

En cuanto a la interpretabilidad, la importancia de variables muestra una concentración destacada en previous_success, con un valor de 0,5361, seguida por active_loan_products_count, con 0,0784. También aparecen entre las variables relevantes is_single, age, is_married, days_since_previous_contact, previous, balance y algunas categorías de ocupación y educación. Esto indica que el historial de resultados positivos en contactos anteriores tiene una contribución especialmente alta en las predicciones del modelo. Sin embargo, la importancia de una variable no demuestra una relación causal ni significa que modificarla produzca directamente una mayor probabilidad de suscripción. Además, dado el peso considerable de previous_success, es importante verificar que esta información esté disponible antes del momento real de predicción y que corresponda exclusivamente a resultados de contactos anteriores, sin incorporar información del resultado que se pretende anticipar.

Finalmente, el registro de experimentos en MLflow permitió mantener la trazabilidad de los modelos evaluados, sus parámetros, métricas y artefactos. Se registraron las variantes de boosting incorporadas al proyecto, los resultados de la validación agrupada y el modelo final seleccionado. En particular, el experimento bank-marketing-modelamiento-integral contiene las ejecuciones de comparación de modelos y el registro final de xgboost_sin_balanceo, mientras que bank-marketing-boosting-balanceo conserva las ejecuciones individuales de las variantes de Gradient Boosting y XGBoost. El registro del modelo final incluye el umbral de 0,2, el criterio de selección basado en Lift@10 medio en validación agrupada, la estrategia comercial principal Top 10 %, la alternativa Top 20 % y la advertencia sobre la exposición histórica del conjunto de prueba. Esta trazabilidad facilita documentar y reproducir las decisiones del proceso de modelamiento.

En conjunto, los resultados respaldan la elección de XGBoost sin balanceo como modelo candidato final, con un umbral de 0,2 para la clasificación binaria y una estrategia comercial principal de priorización Top 10 %, dejando Top 20 % como alternativa de mayor cobertura. La selección responde al objetivo de priorizar clientes y optimizar la asignación de esfuerzos comerciales, no simplemente de maximizar una métrica general de clasificación. Su desempeño deberá seguirse en operación y validarse con datos nuevos, especialmente para confirmar la estabilidad de las métricas y la disponibilidad efectiva de las variables utilizadas en el momento de realizar las predicciones.